In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RAW_DIR = Path('../data/raw')
IMG_DIR = RAW_DIR / 'train_images'
PROC_DIR = Path('../data/processed')

In [ ]:
df = pd.read_csv(RAW_DIR / 'train.csv')
print(f'Total samples : {len(df):,}')
print(f'Positives     : {df["target"].sum():,}  ({df["target"].mean()*100:.2f}%)')
print(f'Columns       : {list(df.columns)}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['target'].value_counts()
axes[0].bar(['Benign (0)', 'Malignant (1)'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

# Age distribution by class
df.groupby('target')['age_approx'].plot.hist(
    bins=20, alpha=0.7, ax=axes[1], legend=True, color=['steelblue', 'tomato']
)
axes[1].set_title('Age Distribution by Class')
axes[1].set_xlabel('Age (approx)')
axes[1].legend(['Benign', 'Malignant'])

plt.tight_layout()
plt.savefig('../notebooks/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sex distribution per class
sex_counts = df.groupby(['target', 'sex']).size().unstack(fill_value=0)
sex_counts.T.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Sex Distribution by Class')
axes[0].set_xlabel('Sex')
axes[0].tick_params(axis='x', rotation=0)

# Anatomical site per class
site_pos_rate = df.groupby('anatom_site_general_challenge')['target'].mean().sort_values(ascending=False)
site_pos_rate.plot(kind='bar', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Positive Rate by Anatomical Site')
axes[1].set_ylabel('Malignant Fraction')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Sample image grid — 5 benign and 5 malignant
benign_names = df[df['target'] == 0]['image_name'].sample(5, random_state=42).tolist()
malignant_names = df[df['target'] == 1]['image_name'].sample(5, random_state=42).tolist()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for col, name in enumerate(benign_names):
    img = Image.open(IMG_DIR / f'{name}.jpg').resize((256, 256))
    axes[0, col].imshow(img)
    axes[0, col].set_title('Benign', fontsize=9)
    axes[0, col].axis('off')

for col, name in enumerate(malignant_names):
    img = Image.open(IMG_DIR / f'{name}.jpg').resize((256, 256))
    axes[1, col].imshow(img)
    axes[1, col].set_title('Malignant', fontsize=9, color='red')
    axes[1, col].axis('off')

plt.suptitle('Sample Dermoscopy Images', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Missing value summary
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Unique patients:', df['patient_id'].nunique() if 'patient_id' in df.columns else 'N/A')